# Multi-head attention in PyTorch

*A companion to the blog post “Multi-head attention: many ways of looking at one sentence” (Part 6 of the Transformer on-ramp series).*

Build a multi-head attention layer, follow the tensor shapes, and compare it with one big head.

In [ ]:
import torch

def scaled_dot_product_attention(Q, K, V, mask=None):
    # Q, K, V: (batch, heads, seq, d_k)
    scores = Q @ K.transpose(-2, -1) / (K.shape[-1] ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = scores.softmax(dim=-1)
    return weights @ V, weights

## The layer

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, h):
        super().__init__()
        assert d_model % h == 0
        self.h, self.dk = h, d_model // h
        self.qkv = torch.nn.Linear(d_model, 3 * d_model)   # one matrix makes Q, K and V
        self.out = torch.nn.Linear(d_model, d_model)       # W_O

    def forward(self, x, causal=True):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.h, self.dk).transpose(1, 2)   # (B, h, T, dk)
        q, k, v = split(q), split(k), split(v)
        mask = torch.tril(torch.ones(T, T)) if causal else None
        o, w = scaled_dot_product_attention(q, k, v, mask)
        o = o.transpose(1, 2).reshape(B, T, D)             # concatenate the heads
        return self.out(o), w

## Follow the shapes

In [ ]:
torch.manual_seed(0)
m = MultiHeadAttention(d_model=32, h=4)
x = torch.randn(2, 6, 32)                  # (batch, tokens, d_model)
q, k, v = m.qkv(x).chunk(3, dim=-1)
print('after the big projection :', tuple(q.shape))
split = lambda t: t.view(2, 6, 4, 8).transpose(1, 2)
print('split into heads         :', tuple(split(q).shape), ' (batch, heads, tokens, d_k)')
y, w = m(x)
print('layer output             :', tuple(y.shape), ' same as the input')
print('attention weights        :', tuple(w.shape), ' one table per head')

## Check each head against PyTorch
The per-head attention should match PyTorch's built-in function.

In [ ]:
ref = torch.nn.functional.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
mine, _ = scaled_dot_product_attention(split(q), split(k), split(v), torch.tril(torch.ones(6, 6)))
print('matches PyTorch:', torch.allclose(mine, ref, atol=1e-5))

## Is it more expensive than one big head?

In [ ]:
def n_weights(layer):
    return sum(p.numel() for n, p in layer.named_parameters() if 'weight' in n)

print('1 head  :', n_weights(MultiHeadAttention(32, 1)))
print('4 heads :', n_weights(MultiHeadAttention(32, 4)))
print('8 heads :', n_weights(MultiHeadAttention(32, 8)))

The count does not change: more heads just slice the same width differently. **Try it:** change `d_model` to 64 and `h` to 16. What is `d_k`?

### Want the full guided version?

This notebook is a small taster from **Chapter 1 of the course *Generative AI and LLMs with Python*** (course-2-generative-ai), one of 4 world-class courses on [aibits.ecubeanalytics.com](https://aibits.ecubeanalytics.com/?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01).

**The full guided lab, quizzes and projects are in the course.** Chapter 1 is part of the free first module:

- [Read Chapter 1 free (sign up)](https://aibits.ecubeanalytics.com/main/document/genai_gate.php?c=course-2-generative-ai&f=chapters/course-2-generative-ai/01-on-ramp-neurons-to-the-transformer.html&utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01)
- [See the course demo](https://aibits.ecubeanalytics.com/main/document/genai_public/GENAILLMPYTHON.html?utm_source=colab&utm_medium=notebook&utm_campaign=genai_ch01)
